# From Pipeline to Agent

In the **last class** we built an NL-to-SQL pipeline. It worked, but it could only do one thing: write SQL and run it. Today we turn it into an **agent**.

## What is the difference?

A **pipeline** is a fixed sequence of steps. We had six of them, in this order, every time:

```
question  ->  generate SQL  ->  validate  ->  execute  ->  summarize  ->  answer
```

An **agent** does not have a fixed sequence. Instead, it has **tools**, and on every turn it decides what to do next:

```
                +--------> tool A (e.g. SQL)
                |
question -> LLM +--------> tool B (e.g. lookup)
                |              |
                |              v
                +<------- observation
                |
                +-> answer (when it has enough)
```

This loop is called **ReAct** (Reason + Act). On every iteration the LLM either:
- emits a tool call (the action), then we run the tool and feed the result back, OR
- emits a final answer (and we stop).

## Why we need this

Look at the questions our pipeline could **not** answer last class:

- "What is the thrust of the rocket we use for crewed missions?"
- "Which of our rockets is reusable?"
- "Compare Falcon Heavy and Starship: payload capacity and our missions on each."

Our SQL database does not have rocket specs (thrust, height, reusability). Those facts live in a different file (`vehicle_specs.json`). So a single SQL query is not enough. The agent will pick which tool to call.


## Setup


In [57]:
# uv add reads pyproject.toml in this folder.
# If you don't have uv: curl -LsSf https://astral.sh/uv/install.sh | sh
!uv add langchain-openai langchain-core pandas

'uv' is not recognized as an internal or external command,
operable program or batch file.


In [58]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

In [59]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

DB = Path("spacex_launches.db")
SCHEMA = Path("schema.md").read_text()
SPECS = json.loads(Path("vehicle_specs.json").read_text())

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. The two data sources

The agent will have access to:

1. **`spacex_launches.db`** - the same SQLite database from last class. One table, `launches`.
2. **`vehicle_specs.json`** - a JSON file of rocket specs (thrust, height, reusability, etc.). Same data shape we'd hit if we were calling a microservice.

Two completely different storage formats. Two different access patterns. The agent has to know which one to ask.


In [60]:
# launches table - structured, query with SQL
con = sqlite3.connect(DB)
df_launches = pd.read_sql_query("SELECT * FROM launches ORDER BY launch_date", con)
con.close()
df_launches.head(5)

,launch_id,mission_name,vehicle,payload_type,payload_kg,destination,success,launch_date,customer
0,1,Falcon 1 Flight 1,Falcon 1,Test,20,LEO,0,2006-03-24,DARPA
1,2,Falcon 1 Flight 4,Falcon 1,Test,165,LEO,1,2008-09-28,SpaceX
2,3,COTS Demo Flight 2,Falcon 9,Cargo,525,ISS,1,2012-05-22,NASA
3,4,CASSIOPE,Falcon 9,Satellite,500,LEO,1,2013-09-29,MDA
4,5,DSCOVR,Falcon 9,Satellite,570,L1,1,2015-02-11,NOAA


In [61]:
# vehicle specs - JSON file, lookup by key
print(json.dumps(SPECS, indent=2)[:900])
print("...")

{
  "Falcon 1": {
    "first_flight": "2006-03-24",
    "height_m": 21,
    "diameter_m": 1.7,
    "mass_kg": 30000,
    "thrust_kn": 454,
    "payload_to_leo_kg": 670,
    "stages": 2,
    "reusable": false,
    "active": false,
    "manufacturer": "SpaceX",
    "notes": "First privately-developed liquid rocket to reach orbit. Retired in 2009."
  },
  "Falcon 9": {
    "first_flight": "2010-06-04",
    "height_m": 70,
    "diameter_m": 3.7,
    "mass_kg": 549054,
    "thrust_kn": 7607,
    "payload_to_leo_kg": 22800,
    "stages": 2,
    "reusable": true,
    "active": true,
    "manufacturer": "SpaceX",
    "notes": "Workhorse rocket. First stage is reusable - lands on a droneship or back at the launch site."
  },
  "Falcon Heavy": {
    "first_flight": "2018-02-06",
    "height_m": 70,
    "diameter_m": 12.2,
    "mass_kg": 1420788,
    "thrust_kn": 22819,
    "payload_to_leo_kg": 638
...


## 2. A question the pipeline cannot answer

> *"What is the thrust of the rocket we use for crewed missions?"*

To answer this, you need to:

1. Find which vehicle is used for crewed missions. **(SQL on `launches`)**
2. Look up the thrust of that vehicle. **(JSON lookup in `vehicle_specs`)**

A pipeline always uses SQL. It can find the vehicle (Falcon 9), but it cannot then go look up its thrust. The agent will do both.


## 3. Define the tools

A **tool** in LangChain is just a Python function with a description. The LLM reads the description and decides when to call it.

The `@tool` decorator wraps the function so the LLM can see its name, arguments, and docstring. We define two:


In [62]:
FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")


@tool
def query_launches(sql: str) -> str:
    """Run a read-only SELECT query on the SQLite `launches` table.

    The launches table has columns:
      launch_id, mission_name, vehicle, payload_type, payload_kg,
      destination, success, launch_date, customer.

    Returns the result rows as a JSON-style list of dicts (truncated to 50 rows).
    Refuses any non-SELECT statement.
    """
    upper = sql.upper()
    for w in FORBIDDEN:
        if w in upper:
            return f"REFUSED: SQL contains {w}"
    if "FROM LAUNCHES" not in upper.replace("\n", " "):
        return "REFUSED: SQL must read from launches"
    if "LIMIT" not in upper and not any(a in upper for a in ("COUNT(", "SUM(", "AVG(", "MIN(", "MAX(", "GROUP BY")):
        sql = sql.rstrip(";\n ") + " LIMIT 50"
    try:
        con = sqlite3.connect(DB)
        con.row_factory = sqlite3.Row
        rows = [dict(r) for r in con.execute(sql).fetchall()]
        con.close()
    except Exception as e:
        return f"SQL ERROR: {e}"
    return json.dumps(rows[:50])


@tool
def vehicle_specs(vehicle: str) -> str:
    """Look up technical specifications for a SpaceX rocket family.

    Allowed values for `vehicle`: 'Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship'.

    Returns the spec sheet as a JSON object: thrust_kn, height_m, mass_kg,
    payload_to_leo_kg, reusable, active, first_flight, manufacturer, notes.
    """
    spec = SPECS.get(vehicle)
    if spec is None:
        return f"UNKNOWN VEHICLE: {vehicle!r}. Allowed: {list(SPECS.keys())}"
    return json.dumps(spec)

@tool
def compare(a: str, b: str, label: str) -> str:
    """Use this tool whenever the user asks to compare two values."""
    return f"{label}: a={a}, b={b}"


tools = [query_launches, vehicle_specs,compare]
TOOLS_BY_NAME = {t.name: t for t in tools}

print("Tools defined:", [t.name for t in tools])





Tools defined: ['query_launches', 'vehicle_specs', 'compare']


## 4. Bind the tools to the LLM

`bind_tools()` tells the LLM which tools exist. The LLM does **not** call the functions itself - it emits a "tool call" object describing which function to run with which arguments. We then run it ourselves in Python and feed the result back. This is what every modern agent framework does under the hood.


In [63]:
agent_llm = llm.bind_tools(tools)

# Quick check - send a simple question and inspect the response
test_response = agent_llm.invoke([
    SystemMessage(content="You answer questions using the tools provided."),
    HumanMessage(content="What is the thrust of Falcon 9?"),
])

print("LLM content:    ", test_response.content)
print("LLM tool_calls: ", test_response.tool_calls)

LLM content:     
LLM tool_calls:  [{'name': 'vehicle_specs', 'args': {'vehicle': 'Falcon 9'}, 'id': 'call_44PLMEfd7bBAeffCvQTmQOPR', 'type': 'tool_call'}]


See? The LLM did **not** answer in text. It returned a `tool_call` saying *"please call `vehicle_specs` with `vehicle='Falcon 9'`"*. We have to actually run the function and feed back the result. That is the agent's job.


## 5. The agent's system prompt

The system prompt is shorter than the pipeline's was. We do not tell the LLM how to write SQL exhaustively - we just hand it tools and tell it when to use which.

We do still include `schema.md` so the LLM knows what columns exist (so it can write good SQL inside the `query_launches` tool call) and what domain terms mean.


In [64]:
AGENT_SYSTEM = f"""You are an analyst with access to two tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.

Use the tools to gather facts. When you have enough information, answer the user
in one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,
say so honestly.

Do not call a tool if you can answer from prior tool results. Do not guess - look up.

Schema for the SQL tool:

{SCHEMA}
"""

print(AGENT_SYSTEM[:600])
print("...")

You are an analyst with access to two tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.

Use the tools to gather facts. When you have enough information, answer the user
in one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,
say so honestly.

Do not call a tool if you can answer from prior tool results. Do not guess - look up.

Schema for the SQL tool:

# Schema for the LLM

This file is what the agent's **system prompt** loads. The DDL in
`seed.sql` is wha
...


## 6. The agent loop (ReAct, written manually)

The whole agent is one Python function with a loop. On each iteration we:

1. Send the conversation so far to the LLM.
2. If the LLM emits a tool call, we run the tool and append its result.
3. If the LLM emits text content with no tool call, that's the final answer.
4. We bail out after a max number of iterations as a safety net.

This is the **same loop** that frameworks like LangGraph, CrewAI, AutoGen wrap up. Writing it ourselves makes the abstraction concrete.


In [65]:
def run_agent(question, max_iters=8, verbose=True):
    messages = [
        SystemMessage(content=AGENT_SYSTEM),
        HumanMessage(content=question),
    ]
    if verbose:
        print(f"USER: {question}\n")

    for step in range(1, max_iters + 1):
        response = agent_llm.invoke(messages)
        messages.append(response)
        print(messages)
        print(response.tool_calls)

        if not response.tool_calls:
            # Final answer
            if verbose:
                print(f"--- step {step}: final answer ---")
                print(response.content)
            return response.content

        # Run each tool the LLM asked for
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]
            if verbose:
                print(f"--- step {step}: action ---")
                print(f"  tool: {name}")
                print(f"  args: {args}")
            tool_fn = TOOLS_BY_NAME[name]
            result = tool_fn.invoke(args)
            if verbose:
                short = result[:240] + ("..." if len(result) > 240 else "")
                print(f"  observation: {short}\n")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Agent hit max iterations without finishing."


# Quick smoke test
_ = run_agent("What is the thrust of Falcon 9?", max_iters=3)

USER: What is the thrust of Falcon 9?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   | the LLM | U

## 7. Trace a multi-tool question end-to-end

Now the question that motivated this whole class:

> *"What is the thrust of the rocket we use for crewed missions?"*

Watch how the agent reasons. It needs to call **both** tools to answer.


In [66]:
_ = run_agent("What is the thrust of the rocket we use for crewed missions?")

USER: What is the thrust of the rocket we use for crewed missions?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n

**Read the trace.** The agent:

1. Called `query_launches` with `SELECT DISTINCT vehicle FROM launches WHERE payload_type = 'Crew'` to find the rocket.
2. Got back `Falcon 9`.
3. Called `vehicle_specs('Falcon 9')` to get the thrust.
4. Composed the final answer using both pieces.

This is the **whole point of an agent**: the LLM decided which tool to call, and in what order, and when it had enough to answer. We did not script it.


## 8. Try more multi-tool questions

Each of these requires a different mix of tool calls. Watch the traces.


In [67]:
_ = run_agent("Which of our rockets is reusable?")

USER: Which of our rockets is reusable?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   | the LLM |

In [68]:
_ = run_agent("How many missions have flown on reusable rockets?")

USER: How many missions have flown on reusable rockets?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.m

In [69]:
_ = run_agent(
    "Compare Falcon Heavy and Starship. Which has higher thrust, "
    "and how many successful missions has each flown?"
)

USER: Compare Falcon Heavy and Starship. Which has higher thrust, and how many successful missions has each flown?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Crea

## 9. Pipeline vs agent on the same question

For SQL-only questions (like the ones from last class), the agent reduces to one tool call. Same answer, slightly more overhead. The agent's value shows up when **the question requires more than one source**.


In [70]:
# A simple SQL-only question - the agent uses one tool
_ = run_agent("How many launches were successful?", verbose=True)

USER: How many launches were successful?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   | the LLM 

In [71]:
# A specs-only question - the agent uses one tool, the OTHER one
_ = run_agent("How tall is Starship?", verbose=True)

USER: How tall is Starship?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   | the LLM | Understand 

## 10. Failure modes


### Failure 1: the agent makes things up when both tools come back empty

Ask about something neither tool can answer. The agent SHOULD say it doesn't know. Watch.


In [72]:
_ = run_agent("What is the diameter of the Saturn V rocket?", max_iters=4)

USER: What is the diameter of the Saturn V rocket?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   

### Failure 2: the agent runs in circles

If we give a vague question, the agent may keep calling tools without converging. The `max_iters` safety valve catches it.


In [73]:
_ = run_agent("Tell me everything interesting about our missions.", max_iters=4)

USER: Tell me everything interesting about our missions.

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.

In [74]:
_ = run_agent("What is the diameter of the Saturn V rocket?" )

USER: What is the diameter of the Saturn V rocket?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   

In [75]:
_ = run_agent("What is the thrust of the rocket we use for crewed missions?" )

USER: What is the thrust of the rocket we use for crewed missions?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n

In [76]:
_ = run_agent( "Compare Falcon Heavy and Starship. Which has higher thrust?")

USER: Compare Falcon Heavy and Starship. Which has higher thrust?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n|

In [77]:
_ = run_agent( "Compare Falcon Heavy and Starship thrust?")

USER: Compare Falcon Heavy and Starship thrust?

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`   | t

### Failure 3: validator catches dangerous SQL

The same `validate` logic from last class lives inside the `query_launches` tool. The agent cannot bypass it.


In [78]:
_ = run_agent("Delete the failed launches from the database.", max_iters=3)

USER: Delete the failed launches from the database.

[SystemMessage(content='You are an analyst with access to two tools:\n\n1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.\n2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.\n\nUse the tools to gather facts. When you have enough information, answer the user\nin one short paragraph. Cite numbers exactly. If a tool returns an error or no rows,\nsay so honestly.\n\nDo not call a tool if you can answer from prior tool results. Do not guess - look up.\n\nSchema for the SQL tool:\n\n# Schema for the LLM\n\nThis file is what the agent\'s **system prompt** loads. The DDL in\n`seed.sql` is what SQLite loads. They serve different audiences:\n\n| File          | Reader  | Job                                              |\n|---------------|---------|--------------------------------------------------|\n| `seed.sql`    | SQLite  | Create the table, store the rows.                |\n| `schema.md`  

## 11. What this agent still cannot do

We solved the multi-source problem. We did not solve everything:

| Gap                                  | What you'd add                                                  |
|--------------------------------------|------------------------------------------------------------------|
| No memory across runs                 | Persist the messages list to disk per user/conversation.         |
| No unstructured docs (press releases) | A vector-search tool over a document store.                      |
| No real-time data                     | A web-search tool, or live API calls.                             |
| No charts                              | A `make_chart(spec)` tool that returns an image.                  |
| Confident hallucinations               | Stricter prompts + a "I'm not sure" escape hatch in the agent.   |
| Cost / latency creep                   | Tool-call caching + max-iters tuning + smaller models for retries.|

Adding tools is cheap; the hard part is choosing which tools are worth the agent's attention budget.


## 12. Try it yourself

1. Add a third tool: `compare(a, b, label)` that just returns `f"{label}: a={a}, b={b}"`. Ask a comparison question. Does the agent use it?
2. Add a fourth tool that the agent should *never* need to use (e.g. `random_number()`). Does the agent leave it alone?
3. Edit `vehicle_specs.json` and add a new key (`"crew_capacity"`). Ask a question that needs that key. Does the agent pull it out correctly?
4. Drop `temperature` from 0 to 0.7 and re-run the comparison question three times. Are the traces identical? What changed?
5. Replace `gpt-4o-mini` with `gpt-3.5-turbo` and run the multi-tool question. Does the agent still call both tools, or does it skip one?
6. Add `print(messages)` after the loop. How does the conversation grow with each step?
